# Document Parsing -- PDF, HTML, and Text

This file teaches you how to extract text from different file types. 
By the end, you will know how to read PDFs, parse HTML pages, extract tables, and load plain text and Markdown files.

## Setup

We need two libraries:
- **PyMuPDF** (imported as `fitz`): reads PDF files.
- **BeautifulSoup** (from `bs4`): reads HTML files.

Run the cell below to install them and imoprt what we need. 

In [1]:
!pip install ipykernel


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -q pymupdf beautifulsoup4


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import re
import fitz 
from bs4 import BeautifulSoup

**What happened:** We installed PyMuPDF and BeautifulSoup, then imported them.

## PDF Parsing with PyMuPDF

**What it does:** Opens a PDF file and pulls out all the text.

**When to use it:** When your documents are PDF files with selectable text (not scanned images).

**Steps:**
1. Create a sample PDF so we have something to work with. 
2. Open the PDF and read text from every page. 
3. Print the result. 

First, let us create a small sample PDF. 

In [7]:
os.makedirs("./tmp", exist_ok=True)

In [8]:
pdf_path = "./tmp/VyshaliSamplePDF.pdf"

In [9]:
doc = fitz.open()

In [10]:
page = doc.new_page()

In [11]:
text = (
    "RAG Document Guide\n\n"
    "Chapter 1: Introduction\n"
    "RAG combines retrieval with generation.\n\n"
    "Chapter 2: Steps\n"
    "1. Parse documents  2. Chunk  3. Embed  4. Retrieve"
)

In [12]:
page.insert_text(
    (72,72),
    text=text,
    fontsize=11
)

7

In [13]:
doc.save(pdf_path)

In [14]:
doc.close()

**What happened:** We created a one-page PDF with some example text and saved it to disk

### Extract All Text from a PDF

In [27]:
doc = fitz.open(pdf_path)

print(doc, "\n")
print(len(doc), "\n")
print(doc[0], "\n")
print(doc[0].get_text(), "\n")

Document('./tmp/VyshaliSamplePDF.pdf') 

1 

page 0 of <./tmp/VyshaliSamplePDF.pdf, doc# 13> 

RAG Document Guide
Chapter 1: Introduction
RAG combines retrieval with generation.
Chapter 2: Steps
1. Parse documents  2. Chunk  3. Embed  4. Retrieve
 



In [32]:
full_text = ""

for page_num in range(len(doc)):
    
    print("PAGE NUMBER\n", page_num, "\n")
    print("PAGE TEXT\n", doc[page_num].get_text())
    
    full_text += doc[page_num].get_text()

PAGE NUMBER
 0 

PAGE TEXT
 RAG Document Guide
Chapter 1: Introduction
RAG combines retrieval with generation.
Chapter 2: Steps
1. Parse documents  2. Chunk  3. Embed  4. Retrieve



In [33]:
doc.close()

In [46]:
print("Number of Characters:", len(full_text), "\n")

print(f"FULL TEXT: \n {full_text[:75]}...")

Number of Characters: 152 

FULL TEXT: 
 RAG Document Guide
Chapter 1: Introduction
RAG combines retrieval with gene...


**What happened:** We opened the PDF, looped through every page, and collected all the text into one string.

In [47]:
doc = fitz.open(pdf_path)

In [49]:
pages = []

for i in range(len(doc)):
    
    page_text = doc[i].get_text()

    page_text = page_text.strip()

    number_of_characters = len(doc[i].get_text())

    page_info = {
        "page": i+1,
        "text": page_text,
        "chars": number_of_characters
    }

    pages.append(page_info)


doc.close()



In [54]:
import json

for p in pages:
    print(p)

{'page': 1, 'text': 'RAG Document Guide\nChapter 1: Introduction\nRAG combines retrieval with generation.\nChapter 2: Steps\n1. Parse documents  2. Chunk  3. Embed  4. Retrieve', 'chars': 152}


**What happened:** We stored each page's text in a dictionary along with its page number and character count. This metadata is useful when you need to trace text back to its source page.

### Block-Level Extraction 

**What it does:** Extracts text in blocks (group of text positioned together on the page) instead of as one big string.

**When to use it:** When you need to know the layout of the page. 

In [55]:
doc = fitz.open(pdf_path)

In [56]:
page = doc[0]

In [57]:
blocks = page.get_text("blocks")

print(blocks)

[(72.0, 60.17499923706055, 181.427978515625, 75.28900146484375, 'RAG Document Guide\n', 0, 0), (72.0, 90.40303039550781, 268.844970703125, 120.63104248046875, 'Chapter 1: Introduction\nRAG combines retrieval with generation.\n', 1, 0), (72.0, 135.7450714111328, 332.4579772949219, 165.97308349609375, 'Chapter 2: Steps\n1. Parse documents  2. Chunk  3. Embed  4. Retrieve\n', 2, 0)]


In [59]:
for i, block in enumerate(blocks):
    block_text = block[4]
    block_text = block_text.strip()
    print(f"BLOCK {i}\n", block_text, "\n")

BLOCK 0
 RAG Document Guide 

BLOCK 1
 Chapter 1: Introduction
RAG combines retrieval with generation. 

BLOCK 2
 Chapter 2: Steps
1. Parse documents  2. Chunk  3. Embed  4. Retrieve 



In [60]:
doc.close()

**What happened:** Each block is a reactangle of text on the page. We printed the text content of each block. 